In [17]:
import pandas as pd
import os

UOB_PATH = r"C:\Users\hp\Desktop\uob_papers.csv"
COMBINED_PATH = r"c:\Users\hp\Desktop\Research-System\training\datasets\dataset_combined.csv"

In [3]:
import chardet

with open(UOB_PATH, 'rb') as f:
    raw = f.read(10000)  # read first 10k bytes
    result = chardet.detect(raw)
    print(result)

{'encoding': 'ISO-8859-1', 'confidence': 0.7637707779836423, 'language': 'en', 'mime_type': 'text/plain'}


In [4]:
df = pd.read_csv(UOB_PATH, encoding='ISO-8859-1')

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print()
display(df.head(3))

Shape: (270, 7)
Columns: ['title', 'abstract', 'main_label', 'sub_label', 'source', 'relevant', 'is_duplicate']



,title,abstract,main_label,sub_label,source,relevant,is_duplicate
0,Software Engineering for Developing a Cloud Co...,The aim of this article proposes an innovative...,Computer Science,Artificial Intelligence,uob,Yes,No
1,Predicting Software Faults Based on K-Nearest ...,Software defect prediction is one of the most ...,Computer Science,Machine Learning,uob,Yes,No
2,Mining program Learning Outcomes for Students ...,The purpose of the research presented in this ...,Computer Science,Machine Learning,uob,Yes,No


In [5]:
# Drop the extra columns
df = df[['title', 'abstract', 'main_label', 'sub_label']]

print(f"New shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
display(df.head(3))

New shape: (270, 4)
Columns: ['title', 'abstract', 'main_label', 'sub_label']


,title,abstract,main_label,sub_label
0,Software Engineering for Developing a Cloud Co...,The aim of this article proposes an innovative...,Computer Science,Artificial Intelligence
1,Predicting Software Faults Based on K-Nearest ...,Software defect prediction is one of the most ...,Computer Science,Machine Learning
2,Mining program Learning Outcomes for Students ...,The purpose of the research presented in this ...,Computer Science,Machine Learning


In [6]:
print("Unique main_label values:")
print(df['main_label'].unique())
print("\nUnique sub_label values:")
print(df['sub_label'].unique())
print("\nAll main-sub combinations:")
display(df[['main_label', 'sub_label']].drop_duplicates().sort_values(['main_label', 'sub_label']))

Unique main_label values:
['Computer Science' nan]

Unique sub_label values:
['Artificial Intelligence' 'Machine Learning' nan
 'Natural Language Processing' 'Data Structures & Algorithms'
 'Cryptography & Security' 'Robotics']

All main-sub combinations:


,main_label,sub_label
0,Computer Science,Artificial Intelligence
12,Computer Science,Cryptography & Security
9,Computer Science,Data Structures & Algorithms
1,Computer Science,Machine Learning
4,Computer Science,Natural Language Processing
46,Computer Science,Robotics
3,NaN,NaN


In [7]:
print("Null counts per column:")
print(df.isnull().sum())
print(f"\nTotal rows: {len(df)}")

Null counts per column:
title         36
abstract      36
main_label    36
sub_label     36
dtype: int64

Total rows: 270


In [8]:
# Show rows where title is NaN
nan_rows = df[df['title'].isna()]
print(f"Number of NaN rows: {len(nan_rows)}")
display(nan_rows.head(10))

Number of NaN rows: 36


,title,abstract,main_label,sub_label
3,NaN,NaN,NaN,NaN
10,NaN,NaN,NaN,NaN
15,NaN,NaN,NaN,NaN
62,NaN,NaN,NaN,NaN
117,NaN,NaN,NaN,NaN
120,NaN,NaN,NaN,NaN
129,NaN,NaN,NaN,NaN
143,NaN,NaN,NaN,NaN
160,NaN,NaN,NaN,NaN
169,NaN,NaN,NaN,NaN


In [9]:
# Drop rows where title is NaN (completely empty rows)
df = df.dropna(subset=['title']).copy()

print(f"Shape after dropping NaN rows: {df.shape}")
print(f"Rows dropped: 36 → {len(df)} remaining")

Shape after dropping NaN rows: (234, 4)
Rows dropped: 36 → 234 remaining


In [10]:
# Check for duplicate titles within the UOB dataset itself
dupes = df[df.duplicated(subset=['title'], keep=False)]
print(f"Number of duplicated titles within UOB: {len(dupes)}")
if len(dupes) > 0:
    display(dupes.sort_values('title').head(10))

Number of duplicated titles within UOB: 0


In [11]:
# Check for missing or empty abstracts
missing_abstracts = df['abstract'].isna() | (df['abstract'].str.strip() == '')
print(f"Rows with missing/empty abstract: {missing_abstracts.sum()}")

# Optional: show them to decide if we keep or drop
if missing_abstracts.sum() > 0:
    display(df[missing_abstracts][['title', 'abstract']].head(10))

Rows with missing/empty abstract: 0


In [12]:
import re

def contains_latex(text):
    """Quick check: does the text contain any LaTeX control sequences?"""
    if not isinstance(text, str):
        return False
    # look for backslash-command, math-mode $, or curly braces with content
    return bool(re.search(r'\\[a-zA-Z]+|\$|\{.*\}', text))

# Flag rows that have LaTeX in either title or abstract
latex_mask = df['title'].apply(contains_latex) | df['abstract'].apply(contains_latex)
latex_rows = df[latex_mask]

print(f"Rows with potential LaTeX: {len(latex_rows)} out of {len(df)}")

if len(latex_rows) > 0:
    print("\nSample (title + first 200 chars of abstract):")
    for _, row in latex_rows.head(10).iterrows():
        print(f"\nTitle: {row['title']}")
        abstract = row['abstract'] if isinstance(row['abstract'], str) else ''
        print(f"Abstract: {abstract[:200]}")

Rows with potential LaTeX: 0 out of 234


In [18]:
combined = pd.read_csv(COMBINED_PATH)

# Normalize titles for matching (lowercase, strip)
uob_titles = df['title'].str.lower().str.strip()
combined_titles = combined['title'].str.lower().str.strip()

# Find overlapping titles
overlap = uob_titles.isin(combined_titles)
print(f"UOB papers already in combined dataset: {overlap.sum()} out of {len(df)}")

if overlap.sum() > 0:
    print("\nOverlapping titles:")
    display(df[overlap][['title', 'main_label', 'sub_label']].head(10))
else:
    print("No overlap found — all UOB papers are new.")

UOB papers already in combined dataset: 1 out of 234

Overlapping titles:


,title,main_label,sub_label
187,A Three-Stage Layer-Based Heuristic to Solve t...,Computer Science,Artificial Intelligence


In [19]:
# Drop the row where title matches the overlapping one (index 187)
df_clean = df[~df.index.isin([187])].copy()

print(f"Rows before: {len(df)}")
print(f"Rows after dropping overlap: {len(df_clean)}")

Rows before: 234
Rows after dropping overlap: 233


In [20]:
# Drop the 'source' column from combined if it's still there
if 'source' in combined.columns:
    combined = combined.drop(columns=['source'])

# Concatenate
merged = pd.concat([combined, df_clean], ignore_index=True)

# Shuffle
merged = merged.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"New combined shape: {merged.shape}")
print(f"Expected: {len(combined)} + {len(df_clean)} = {len(combined) + len(df_clean)}")

New combined shape: (839875, 4)
Expected: 839642 + 233 = 839875


In [21]:
print("Main category distribution:")
display(merged['main_label'].value_counts().reset_index(name='count'))

print("\nSubcategory distribution:")
display(merged['sub_label'].value_counts().reset_index(name='count'))

Main category distribution:


,main_label,count
0,Computer Science,120231
1,Biology & Medicine,119997
2,Mathematics,119991
3,Physics,119988
4,Chemistry,119935
5,Engineering,119904
6,Economics & Business,119829



Subcategory distribution:


,sub_label,count
0,Machine Learning,20091
1,Artificial Intelligence,20056
2,Cryptography & Security,20042
3,Natural Language Processing,20021
4,Data Structures & Algorithms,20015
5,Robotics,20006
6,Astrophysics,20000
7,Neuroscience,20000
8,Analysis,20000
9,Condensed Matter,20000


In [22]:
SAVE_PATH = r"C:\Users\hp\Desktop\final_dataset.csv"  # adjust to your preferred location
merged.to_csv(SAVE_PATH, index=False, encoding='utf-8')
print(f"Saved {len(merged)} papers to {SAVE_PATH}")

Saved 839875 papers to C:\Users\hp\Desktop\final_dataset.csv
